# Notebook 29 — MJO Moisture Preprocessing (match nb13 exactly)
**Project:** ENSO-BSISO SSL — MJO moisture-constraint experiment
**Author:** Jiayi (jh9141@nyu.edu)

Turns the raw moisture/low-wind fields (nb28) into anomaly arrays that are **bit-for-bit comparable**
to `X_MJO` (nb13): meridional average 15S-15N -> 3-harmonic annual cycle (base 1979-2001) ->
120-day preceding running mean -> global temporal-std normalize. Outputs are aligned to the **same
date order** as `labels_aligned_mjo.csv` / `X_MJO` / `mjo_rmm_own_pcs.npy`, so row i matches everywhere.

Produces three meridionally-averaged anomaly fields `(N_days, n_lon)`:
- **`q_col`** = total column water vapour (moisture-mode variable)
- **`q_low`** = 1000-700 hPa integrated specific humidity (skeleton variable)
- **`div_low`** = 1000/925 hPa horizontal divergence (BL convergence; negative = convergence)

---

## Cell 1 — Mount + paths + nb13 pipeline functions (verbatim)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR       = f'{PROJECT_DIR}/MJO'
RAW_DIR       = f'{MJO_DIR}/moisture_constraints/data/raw'
PROCESSED_DIR = f'{MJO_DIR}/data/processed'                 # X_MJO, labels, longitudes live here
OUT_DIR       = f'{MJO_DIR}/moisture_constraints/data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# ---- nb13 anomaly pipeline (copied verbatim so moisture matches X_MJO) ----
BASE_START, BASE_END, N_HARMONICS, PERIOD, WINDOW = 1979, 2001, 3, 365, 120

def build_fourier_features(d, K=3, P=365):
    feats = [np.ones(len(d))]
    for k in range(1, K + 1):
        feats.append(np.cos(2*np.pi*k*d/P)); feats.append(np.sin(2*np.pi*k*d/P))
    return np.column_stack(feats)

def remove_annual_cycle_harmonic(field_2d, doys, base_mask, K=3, P=365):
    T, nx = field_2d.shape
    udoys = np.unique(doys); clim = np.zeros((len(udoys), nx))
    for i, d in enumerate(udoys):
        ids = np.where((doys == d) & base_mask)[0]
        if len(ids): clim[i] = field_2d[ids].mean(axis=0)
    coeffs, *_ = np.linalg.lstsq(build_fourier_features(udoys, K, P), clim, rcond=None)
    smooth = (build_fourier_features(doys, K, P) @ coeffs).astype(np.float32)
    return field_2d - smooth

def remove_running_mean(anom_2d, dates, window=120):
    df = pd.DataFrame(anom_2d.astype(np.float64), index=dates)
    rm = df.rolling(window=window, min_periods=1, closed='left').mean().values
    res = (anom_2d - rm).astype(np.float32)
    nan = np.where(np.isnan(res).any(axis=1))[0]
    if len(nan): res[nan] = anom_2d[nan]
    return res

def global_std_normalize(iso_2d, base_mask):
    s = float(np.nanstd(iso_2d[base_mask].ravel()))
    return (iso_2d / s).astype(np.float32), s

def full_pipeline(field_2d, dates, tag):
    years = dates.year.values; doys = dates.day_of_year.values
    bmask = (years >= BASE_START) & (years <= BASE_END)
    a   = remove_annual_cycle_harmonic(field_2d, doys, bmask, N_HARMONICS, PERIOD)
    iso = remove_running_mean(a, dates, WINDOW)
    n, s = global_std_normalize(iso, bmask)
    print(f'  {tag}: global_std={s:.4g}  base-period std after norm={np.nanstd(n[bmask]):.3f}  NaN={int(np.isnan(n).sum())}')
    return n, s

print('Paths ready. Pipeline constants:', BASE_START, BASE_END, N_HARMONICS, WINDOW)

## Cell 2 — Load raw moisture/winds + canonical date index

In [ ]:
def load_concat(prefix, required=True):
    fs = sorted(f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR)
                if f.startswith(prefix) and f.endswith('.nc'))
    if not fs:
        if required: raise FileNotFoundError(f'no files for {prefix} in {RAW_DIR}')
        print(f'[optional] no files for {prefix} -> skipping (divergence / BL-convergence diagnostic disabled)')
        return None
    ds = xr.concat([xr.open_dataset(f) for f in fs], dim='valid_time').sortby('valid_time')
    return ds

ds_tcwv = load_concat('TCWV_')
ds_q    = load_concat('qplev_')
ds_uv   = load_concat('uvplev_low_', required=False)     # OPTIONAL: only for BL-convergence (trio)

lats = ds_q.latitude.values; lons = ds_q.longitude.values
print('grid:', len(lats), 'lat x', len(lons), 'lon   levels q:', sorted(ds_q.pressure_level.values.tolist()))
if ds_uv is not None:
    print('  uv levels:', sorted(ds_uv.pressure_level.values.tolist()))

lons_x = np.load(f'{PROCESSED_DIR}/longitudes_mjo.npy')
assert len(lons_x) == len(lons), f'lon count mismatch {len(lons_x)} vs {len(lons)}'
assert np.allclose(np.sort(lons_x), np.sort(lons)), 'longitude values differ from X_MJO'

def nd(ds): return pd.DatetimeIndex(ds.valid_time.values).normalize()
mdates = nd(ds_tcwv).intersection(nd(ds_q)).sort_values()
if ds_uv is not None:
    mdates = mdates.intersection(nd(ds_uv)).sort_values()
print('common moisture days:', len(mdates), mdates[0].date(), '..', mdates[-1].date())

def sel_idx(ds):
    mm = {d:i for i,d in enumerate(nd(ds))}
    return np.array([mm[d] for d in mdates])
iq, it = sel_idx(ds_q), sel_idx(ds_tcwv)
iu = sel_idx(ds_uv) if ds_uv is not None else None

## Cell 3 — Build q_col, q_low, div_low on the full grid

- `q_col` = TCWV (kg m^-2).
- `q_low` = (1/g) integral of q over 1000->700 hPa (kg m^-2), trapezoid in pressure.
- `div_low` = d(u)/dx + d(v)/dy using the 1000/925 layer-mean wind (s^-1); negative = convergence.
  Divergence is computed on the full lat-lon grid **before** meridional averaging so the meridional
  convergence into the 15S-15N band is retained.

In [ ]:
g, R = 9.81, 6.371e6

# q_col = TCWV
qcol_3d = ds_tcwv['tcwv'].values[it].astype(np.float32)                 # (T, lat, lon)

# q_low: vertical integral of q over 1000->700 hPa (ascending), /g
qlev = ds_q.pressure_level.values
order = np.argsort(qlev)
p_pa = (qlev[order] * 100.0).astype(np.float64)
qarr = ds_q['q'].values[iq][:, order, :, :].astype(np.float64)
qlow_3d = (np.trapz(qarr, x=p_pa, axis=1) / g).astype(np.float32)
del qarr

lat_mask = (lats >= -15.0) & (lats <= 15.0)
print('lats kept:', lats[lat_mask], f'({lat_mask.sum()} pts)')
qcol = qcol_3d[:, lat_mask, :].mean(axis=1).astype(np.float32)          # (T, n_lon)
qlow = qlow_3d[:, lat_mask, :].mean(axis=1).astype(np.float32)
del qcol_3d, qlow_3d

# div_low (OPTIONAL): layer-mean (1000,925) wind divergence, full grid -> merid-avg
divl = None
if ds_uv is not None:
    ubl = ds_uv['u'].values[iu].mean(axis=1).astype(np.float64)
    vbl = ds_uv['v'].values[iu].mean(axis=1).astype(np.float64)
    dlon = np.deg2rad(np.abs(lons[1]-lons[0])); dlat = np.deg2rad(np.abs(lats[1]-lats[0]))
    coslat = np.cos(np.deg2rad(lats))[None, :, None]
    dudx = np.gradient(ubl, axis=2) / (R*coslat*dlon)
    dvdy = np.gradient(vbl, axis=1) / (R*dlat)
    div_3d = (dudx + dvdy).astype(np.float32)
    del ubl, vbl, dudx, dvdy
    divl = div_3d[:, lat_mask, :].mean(axis=1).astype(np.float32)
    del div_3d
print('merid-averaged: qcol', qcol.shape, ' qlow', qlow.shape,
      ' divl', None if divl is None else divl.shape)

## Cell 4 — Anomaly pipeline (3-harm annual cycle + 120d running mean + std-norm)

In [ ]:
print('q_col:'); qcol_n, s_qcol = full_pipeline(qcol, mdates, 'q_col')
print('q_low:'); qlow_n, s_qlow = full_pipeline(qlow, mdates, 'q_low')
if divl is not None:
    print('div_low:'); divl_n, s_div = full_pipeline(divl, mdates, 'div_low')
else:
    divl_n, s_div = None, None
    print('div_low: SKIPPED (no uvplev download) -> BL-convergence diagnostic disabled in nb30')

## Cell 5 — Align to `labels_aligned_mjo.csv` order + save

Reindex every field so row i matches `labels_aligned_mjo.csv` / `X_MJO` / `mjo_rmm_own_pcs.npy`.

In [ ]:
labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned_mjo.csv', parse_dates=['date'])
ldates = pd.DatetimeIndex(labels['date']).normalize()
mrow = {d:i for i,d in enumerate(mdates)}
missing = [d for d in ldates if d not in mrow]
assert not missing, f'{len(missing)} label dates missing from moisture (e.g. {missing[:3]})'
idx = np.array([mrow[d] for d in ldates])

qcol_out, qlow_out = qcol_n[idx], qlow_n[idx]
np.save(f'{OUT_DIR}/qcol_mjo_processed.npy', qcol_out)
np.save(f'{OUT_DIR}/qlow_mjo_processed.npy', qlow_out)
if divl_n is not None:
    np.save(f'{OUT_DIR}/divlow_mjo_processed.npy', divl_n[idx])
np.save(f'{OUT_DIR}/longitudes_mjo.npy', lons.astype(np.float32))
labels[['date']].to_csv(f'{OUT_DIR}/moisture_dates.csv', index=False)
meta = {'pipeline':'nb13 (3-harm annual cycle base 1979-2001, 120d preceding running mean, global std-norm)',
        'meridional_band':'15S-15N', 'n_days':int(len(idx)), 'n_lon':int(len(lons)),
        'q_low_layer_hPa':[1000,700], 'div_layer_hPa':[1000,925],
        'div_low_available': bool(divl_n is not None),
        'global_std':{'q_col':s_qcol,'q_low':s_qlow,'div_low':s_div},
        'aligned_to':'labels_aligned_mjo.csv (== X_MJO == mjo_rmm_own_pcs.npy row order)'}
json.dump(meta, open(f'{OUT_DIR}/moisture_preprocess_meta.json','w'), indent=2)
print('Saved to', OUT_DIR, ' (div_low saved:', divl_n is not None, ')')
for f in os.listdir(OUT_DIR):
    print('  ', f, round(os.path.getsize(f'{OUT_DIR}/{f}')/1e6,2),'MB')
print('shapes:', qcol_out.shape, qlow_out.shape)

## Cell 6 — Sanity: phase composites (q vs OLR) + ENSO near-zero check

In [ ]:
import matplotlib.pyplot as plt
X = np.load(f'{PROCESSED_DIR}/X_MJO.npy')          # (N,3,1,180) ch0=u850 ch1=OLR' ch2=u200
olr = X[:,1,0,:]                                    # OLR' anomaly (convection = -olr)
active = (~labels['weak_mjo'].values) & (labels['phase'].between(1,8).values)
phase = labels['phase'].values

fig, axes = plt.subplots(8,1, figsize=(13,12), sharex=True)
fig.suptitle('Phase composites: q_col (green), q_low (orange), -OLR convection (red), u850 (blue)',
             fontweight='bold')
for ax, ph in zip(axes, range(1,9)):
    m = active & (phase==ph)
    if m.sum()==0: continue
    ax.plot(lons, qcol_out[m].mean(0), 'g-', lw=1.3, label='q_col')
    ax.plot(lons, qlow_out[m].mean(0), color='orange', lw=1.1, label='q_low')
    ax.plot(lons, -olr[m].mean(0), 'r--', lw=1.3, label='-OLR (convection)')
    ax.plot(lons, X[m,0,0,:].mean(0), 'b-', lw=0.8, alpha=0.6, label='u850')
    ax.axhline(0,color='k',lw=0.4,alpha=0.4); ax.set_ylabel(f'P{ph}\n(N={int(m.sum())})',fontsize=8)
    if ph==1: ax.legend(fontsize=7, ncol=4)
axes[-1].set_xlabel('Longitude')
plt.tight_layout(); p=f'{OUT_DIR}/moisture_phase_composites.png'
plt.savefig(p, dpi=120, bbox_inches='tight'); plt.show(); print('Saved', p)

print('\nENSO composites (should be near 0 after Lee preprocessing):')
for cat in ['El Nino','Neutral','La Nina']:
    m = labels['enso_category'].values==cat
    print(f'  {cat:8s}: max|q_col|={np.abs(qcol_out[m].mean(0)).max():.3f}  '
          f'max|q_low|={np.abs(qlow_out[m].mean(0)).max():.3f}')

---
## Done!
Outputs in `MJO/moisture_constraints/data/processed/`: `qcol_mjo_processed.npy`,
`qlow_mjo_processed.npy`, `divlow_mjo_processed.npy`, `moisture_dates.csv`, meta.

**Next:** `30_mjo_latent_moisture_diagnostics.ipynb` — composite q/OLR by RMM (and SSL) phase, measure
the moisture-convection phase offset delta-theta, Rossby-Kelvin ratio, BL-convergence lead, all
ENSO-stratified.

---
*DDCS Project | jh9141@nyu.edu*